In [1]:
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset

2026-02-23 09:50:28.767302: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/farhanabdurrahmanmusa/Documents/99 Sidehustle/Pak Jojo/IntentMining/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filepath = "review_dengan_intent.csv"
df = pd.read_csv(filepath)
df.head(5)

,userName,score,content,at,intent
0,Nabila Livia,5,sukaa,2026-02-18 10:36:04,Praise & Gratitude
1,Antok Sumawan,5,ya bagus,2026-02-18 10:32:49,Praise & Gratitude
2,Khayyira Ira,5,aku suka banget sama tiktok ini,2026-02-18 10:32:45,Praise & Gratitude
3,Aksay Subang,1,akun gua entah kenapa di Ben padalah gua kaga ...,2026-02-18 10:31:33,Account Issue
4,kenzo zildane alvaro,1,tolong diperbaiki,2026-02-18 10:30:44,General Request/Complaint


In [3]:
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.intent.value_counts()

Before:  1000
After:  1000


intent
Unlabeled/Noise              277
Praise & Gratitude           271
Account Issue                129
Performance Issue            100
Feature Complaint/Request     90
Technical Bug/Crash           72
General Request/Complaint     38
Monetization/Earning          23
Name: count, dtype: int64

In [4]:
dataset = df.content.values.tolist()

In [5]:
dataset

['sukaa',
 'ya bagus',
 'aku suka banget sama tiktok ini',
 'akun gua entah kenapa di Ben padalah gua kaga ngapa ngapain',
 'tolong diperbaiki',
 '😁',
 'cukup lumayan sih sama aplikasi ini , hanya saja banyak bug , kadang suka keluar sendiri dari aplikasi . terimakasih',
 'membagi inspirasi dan motivasi',
 'Sya kasi bintang 5 biyar lebih bagus lagi hai bosku kenpa titok ya sering elor terus adh aph dengan titok ya',
 'sangat bagus apk nya saya dapat uang juga dari tiktok',
 'mantap',
 'senang bgt',
 'aplikasi jelek',
 'bagus tapi akun ku di block...😭😭',
 'gaseru tiktok sampah saya gabisa join padahal tanggal lahir saya sudah benar',
 'Tiktok ini tidak ngeleg makanya saya download ini',
 'apk nya bagus saya suka, tapi diorang lain mempunyai potongan harga, sedang kan diakun saya ko tidak ada ya, tolong update kan akun saya agar mempunyai potongan harga.....terimakasih',
 'good',
 'tolong bug untuk postingan di atur ulang soalnya setiap muat postingan ngestag di 30% terus',
 'o aj sih😹',

In [6]:
from IndobertEmbedding import IndobertEmbedding

embedding = IndobertEmbedding()
vectors = embedding.getEmbeddings(dataset)

Loading HuggingFace model...
Model Loaded.


In [7]:
vectors[1]

[-0.24239346385002136,
 0.7369614243507385,
 0.8010048270225525,
 0.7975281476974487,
 0.6604104042053223,
 -0.7038561701774597,
 -0.7373388409614563,
 -0.32212498784065247,
 0.14311957359313965,
 -0.09515950083732605,
 -1.521763801574707,
 -0.6150507926940918,
 -0.5980915427207947,
 0.2432849109172821,
 0.16330693662166595,
 -0.463636577129364,
 0.2525262236595154,
 -0.6728891134262085,
 1.1959853172302246,
 0.22371475398540497,
 0.3380160629749298,
 -0.1810867190361023,
 -1.0184473991394043,
 -0.18230003118515015,
 0.4908984303474426,
 -0.27113598585128784,
 1.2375767230987549,
 0.9082168340682983,
 -0.7517940998077393,
 -1.197911024093628,
 1.1902967691421509,
 0.8861282467842102,
 -0.47198009490966797,
 2.3889551162719727,
 -1.0048857927322388,
 1.749250888824463,
 -0.07114163786172867,
 0.14599721133708954,
 -0.64015132188797,
 -0.3623638153076172,
 -0.03726603463292122,
 0.8908575773239136,
 0.6029585599899292,
 -0.7062113881111145,
 -1.4313796758651733,
 -0.1309531331062317,
 -0

In [8]:
%%time
model = ITER_DBSCAN(initial_distance=0.4, initial_minimum_samples=16, delta_distance=0.01, delta_minimum_samples=1, max_iteration=15, algorithm="IndoBERT", metric="euclidean")

CPU times: user 122 μs, sys: 403 μs, total: 525 μs
Wall time: 751 μs


In [9]:
%%time
labels = model.fit_predict(vectors)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


CPU times: user 8.75 s, sys: 1.71 s, total: 10.5 s
Wall time: 1.74 s


In [10]:
df['cluster_ids'] = labels
df.cluster_ids.value_counts()

cluster_ids
-1     864
 1      42
 0      22
 2      19
 3       8
 4       8
 5       7
 6       6
 9       4
 7       4
 8       4
 10      3
 11      3
 12      3
 13      3
Name: count, dtype: int64

In [11]:
df.to_excel("result.xlsx", index=False)

In [12]:
evaluate_dataset = EvaluateDataset(filename=filepath, 
                                   filetype='csv', 
                                   text_column='content', 
                                   target_column='intent')

In [13]:
parameters = [
             {
    "distance": 0.4,           # Ditingkatkan agar cakupan lebih luas
    "minimum_samples":16, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoBERT",
    "metric": "euclidean"
},
{
    "distance": 0.4,           # Ditingkatkan agar cakupan lebih luas
    "minimum_samples":16, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    # "algorithm": "IndoBERT",
    # "metric": "euclidean"
}
]

In [14]:
%%time
results = evaluate_dataset.evaulate_iter_dbscan(parameters)

Loading Tensorflow model....
Model Loaded.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 2/2 [00:02<00:00,  1.39s/it]

CPU times: user 16.5 s, sys: 4.25 s, total: 20.7 s
Wall time: 12 s


In [15]:
result_df = pd.DataFrame.from_dict(results)
result_df

,distance,minimum_samples,delta_distance,delta_minimum_samples,max_iteration,algorithm,metric,time,percentage_labelled,clusters,...,homogeneity_score,completeness_score,normalized_mutual_info_score,adjusted_mutual_info_score,adjusted_rand_score,accuracy,precision,recall,f1,intents
0,0.4,16,0.01,1,15,IndoBERT,euclidean,1.19,13.3,14,...,0.05,0.24,0.08,0.07,-0.02,0.321,34.4,32.1,21.7,3
1,0.4,16,0.01,1,15,NaN,NaN,0.99,5.6,4,...,0.01,0.08,0.03,0.02,-0.01,0.242,12.7,24.2,13.7,2


In [16]:
result_df.to_csv("result_evaluation.csv", index=False)